In [2]:
import pandas as pd

jadwal = pd.read_csv('data/jadwal_mahasiswa.csv')
jadwal.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   student_id    50 non-null     str  
 1   student_name  50 non-null     str  
 2   day           50 non-null     str  
 3   start_time    50 non-null     str  
 4   end_time      50 non-null     str  
 5   course        50 non-null     str  
 6   major         50 non-null     str  
dtypes: str(7)
memory usage: 2.9 KB


In [3]:
import pandas as pd

jadwal = pd.read_csv('data/jadwal_mahasiswa.csv')

# convert ke datetime
jadwal['start_time'] = pd.to_datetime(jadwal['start_time'], format='%H:%M')
jadwal['end_time'] = pd.to_datetime(jadwal['end_time'], format='%H:%M')

# normalize day
jadwal['day'] = jadwal['day'].str.capitalize()

last_class = (
    jadwal
    .groupby(['student_id', 'day'])
    .agg(last_end_time=('end_time', 'max'))
    .reset_index()
)

def analyze_start_time(last_class, start_times):
    results = []

    total_students = last_class['student_id'].nunique()

    for day in last_class['day'].unique():
        df_day = last_class[last_class['day'] == day]

        for start in start_times:
            start_dt = pd.to_datetime(start, format='%H:%M')

            # mahasiswa yang sudah selesai sebelum jam latihan
            available = df_day[df_day['last_end_time'] <= start_dt]['student_id'].nunique()

            results.append({
                "day": day,
                "start_time": start,
                "available": available,
                "total": total_students,
                "availability_ratio": round(available / total_students, 2)
            })

    return pd.DataFrame(results)

start_times = ["14:00", "15:00", "16:00", "17:00"]

result = analyze_start_time(last_class, start_times)

print(result)

          day start_time  available  total  availability_ratio
0      Friday      14:00          8     10                 0.8
1      Friday      15:00          8     10                 0.8
2      Friday      16:00          9     10                 0.9
3      Friday      17:00          9     10                 0.9
4      Monday      14:00          5     10                 0.5
5      Monday      15:00          5     10                 0.5
6      Monday      16:00          8     10                 0.8
7      Monday      17:00          8     10                 0.8
8    Thursday      14:00          4     10                 0.4
9    Thursday      15:00          4     10                 0.4
10   Thursday      16:00          7     10                 0.7
11   Thursday      17:00          7     10                 0.7
12  Wednesday      14:00          6     10                 0.6
13  Wednesday      15:00          6     10                 0.6
14  Wednesday      16:00          8     10             